# 01 — Get the data

In this notebook we load the Kepler "Objects of Interest" (KOI) table from NASA, keep only the signals we know the answer for, and save a smaller, balanced sample to use for training later.

- **CONFIRMED** = a real planet.
- **FALSE POSITIVE** = looked like a planet, but turned out to be something else (for example two stars eclipsing each other).
- We drop **CANDIDATE** rows, because nobody knows yet whether they are planets, so they can't teach the model the right answer.

In [1]:
import io
import pandas as pd

RANDOM_SEED = 42      # fixed seed, so we get the same random pick every time
N_PER_CLASS = 400     # how many rows of each class to keep

## 1. Load the table

The file from the NASA Exoplanet Archive starts with lines beginning with `#` that describe the columns. They aren't data, so we skip them.
(We skip only lines that *start* with `#`, because some text fields inside the data can contain a `#` character.)

In [2]:
with open("../data/koi_table.csv") as f:
    data_lines = [line for line in f if not line.startswith("#")]

koi = pd.read_csv(io.StringIO("".join(data_lines)))
print(koi.shape)
koi.head()

(9564, 141)


,rowid,kepid,kepoi_name,kepler_name,koi_disposition,koi_vet_stat,koi_vet_date,koi_pdisposition,koi_score,koi_fpflag_nt,...,koi_dicco_mdec,koi_dicco_mdec_err,koi_dicco_msky,koi_dicco_msky_err,koi_dikco_mra,koi_dikco_mra_err,koi_dikco_mdec,koi_dikco_mdec_err,koi_dikco_msky,koi_dikco_msky_err
0,1,10797460,K00752.01,Kepler-227 b,CONFIRMED,Done,2018-08-16,CANDIDATE,1.000,0,...,0.200,0.160,0.200,0.170,0.080,0.130,0.310,0.170,0.320,0.160
1,2,10797460,K00752.02,Kepler-227 c,CONFIRMED,Done,2018-08-16,CANDIDATE,0.969,0,...,0.000,0.480,0.390,0.360,0.490,0.340,0.120,0.730,0.500,0.450
2,3,10811496,K00753.01,NaN,CANDIDATE,Done,2018-08-16,CANDIDATE,0.000,0,...,-0.034,0.070,0.042,0.072,0.002,0.071,-0.027,0.074,0.027,0.074
3,4,10848459,K00754.01,NaN,FALSE POSITIVE,Done,2018-08-16,FALSE POSITIVE,0.000,0,...,0.147,0.078,0.289,0.079,-0.257,0.072,0.099,0.077,0.276,0.076
4,5,10854555,K00755.01,Kepler-664 b,CONFIRMED,Done,2018-08-16,CANDIDATE,1.000,0,...,-0.090,0.180,0.100,0.140,0.070,0.180,0.020,0.160,0.070,0.200


## 2. Which dispositions are there?

In [3]:
koi["koi_disposition"].value_counts()

koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64

## 3. Keep only CONFIRMED and FALSE POSITIVE

In [4]:
labeled = koi[koi["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])]
labeled["koi_disposition"].value_counts()

koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
Name: count, dtype: int64

## 4. Drop rows with missing values

Some rows are missing a measurement we need (for example the star's size wasn't known, so the planet's size couldn't be worked out). We drop any row where one of these columns is empty **before** the random pick, so all 800 rows in our sample are complete.

This matters because almost all of the missing values were in FALSE POSITIVE rows. If we left them in, the model could learn the shortcut "missing value = false positive" instead of learning anything about planets.

In [5]:
needed = ["koi_period", "koi_time0bk", "koi_duration", "koi_depth", "koi_srad", "koi_prad"]

clean = labeled.dropna(subset=needed)

before = labeled["koi_disposition"].value_counts()
after = clean["koi_disposition"].value_counts()
pd.DataFrame({"before": before, "after": after, "dropped": before - after})

,before,after,dropped
koi_disposition,,,
FALSE POSITIVE,4839,4582,257
CONFIRMED,2748,2746,2


## 5. Randomly pick 400 of each

Taking the same number from each group gives a *balanced* dataset, so the model can't score well just by always guessing the bigger group.

In [6]:
columns = [
    "kepid", "kepoi_name", "koi_disposition",
    "koi_period", "koi_time0bk", "koi_duration",
    "koi_depth", "koi_srad", "koi_prad",
]

confirmed = clean[clean["koi_disposition"] == "CONFIRMED"].sample(N_PER_CLASS, random_state=RANDOM_SEED)
false_pos = clean[clean["koi_disposition"] == "FALSE POSITIVE"].sample(N_PER_CLASS, random_state=RANDOM_SEED)

selected = pd.concat([confirmed, false_pos])[columns].reset_index(drop=True)
selected["koi_disposition"].value_counts()

koi_disposition
CONFIRMED         400
FALSE POSITIVE    400
Name: count, dtype: int64

Check: there should be **no** missing values left in the columns we kept.

In [7]:
missing = selected.isna().sum()
print(missing)
assert missing.sum() == 0, "There are still missing values!"
print("\nNo missing values left.")

kepid              0
kepoi_name         0
koi_disposition    0
koi_period         0
koi_time0bk        0
koi_duration       0
koi_depth          0
koi_srad           0
koi_prad           0
dtype: int64

No missing values left.


In [8]:
selected.head(10)

,kepid,kepoi_name,koi_disposition,koi_period,koi_time0bk,koi_duration,koi_depth,koi_srad,koi_prad
0,6768394,K02086.01,CONFIRMED,7.132947,135.992590,4.3680,145.0,1.307,1.64
1,11612280,K04304.01,CONFIRMED,9.406136,137.707300,4.8820,79.5,1.268,1.14
2,9907129,K03127.01,CONFIRMED,9.705693,140.370900,7.7020,140.8,1.385,2.07
3,11963206,K02820.01,CONFIRMED,3.009125,131.626960,2.2280,191.4,0.893,1.22
4,8934103,K04009.01,CONFIRMED,175.138920,296.037780,10.7670,634.5,1.241,3.06
5,9031209,K03316.01,CONFIRMED,47.056848,161.058640,5.2232,1531.2,0.740,2.84
6,7288306,K03443.01,CONFIRMED,96.169569,170.258100,7.4960,894.5,0.702,2.07
7,6541920,K00157.03,CONFIRMED,31.995525,154.160745,4.2817,1376.1,1.046,4.12
8,7663405,K01519.01,CONFIRMED,5.144365,133.830440,2.7030,310.7,0.904,1.55
9,9480189,K00941.02,CONFIRMED,2.382653,170.666260,1.8888,705.4,0.973,2.74


## 6. Save

In [9]:
selected.to_csv("../data/kepler_selected.csv", index=False)
print("Saved", len(selected), "rows to data/kepler_selected.csv")

Saved 800 rows to data/kepler_selected.csv


## What each column means

| Column | In simple words |
|---|---|
| `kepid` | The ID number of the **star** in the Kepler catalog. One star can have several KOIs (several possible planets). We'll use it later to download that star's brightness data (light curve). |
| `kepoi_name` | The name of this **signal** (the possible planet), like `K00752.01`. The number before the dot is the star, the part after is which signal around that star (.01, .02, …). |
| `koi_disposition` | The answer we want the model to learn: **CONFIRMED** (real planet) or **FALSE POSITIVE** (not a planet). |
| `koi_period` | The **orbital period** in days: how long between one dip in the star's brightness and the next, i.e. how long the "planet" takes to go once around its star. |
| `koi_time0bk` | The time of the **first dip** (middle of the first transit) that Kepler saw, in days, counted in Kepler's own clock (BKJD = Julian Date − 2,454,833). Together with the period, it tells us exactly where every dip is in the data. |
| `koi_duration` | How **long each dip lasts**, in hours: the time the object takes to cross in front of the star. |
| `koi_depth` | How **deep the dip is**: how much the star's light drops during the transit, in parts per million (ppm). 10,000 ppm = 1% dimmer. Bigger objects block more light. Very deep dips often mean it's actually another star, not a planet. |
| `koi_srad` | The **radius of the star**, measured in Sun-radii (1.0 = same size as our Sun). |
| `koi_prad` | The estimated **radius of the planet**, measured in Earth-radii (1.0 = Earth-sized; Jupiter is about 11). It is calculated from the depth and the star's size. Huge values (much bigger than Jupiter) are a hint that it's not really a planet. |
